# 01 — Data Generation & Validation

Membangun empat dataset training (G1/G2/G3/AO) dan test set dari GSM8K, dengan
rationale tiga tingkat kerincian yang dihasilkan satu teacher.

**Prasyarat**
1. Settings → **Internet: ON**
2. Add-ons → Secrets → `TOKENROUTER_API_KEY`
3. Accelerator: **None** — notebook ini tidak memakai GPU

Teacher `moonshotai/kimi-k3-free` berjalan di tier gratis yang tidak menjamin
konkurensi. Cell generasi bersifat resumable: hasil di-append per record, dan
record yang sudah berhasil dilewati saat dijalankan ulang.

In [107]:
!pip install -q datasets

## Config

Semua konstanta di satu tempat. Ini variabel kontrol eksperimen — jangan diubah
setelah generasi jalan.

In [108]:
import os, re, json, time, random, threading, requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

SEED     = 42
N_TRAIN  = 1500
N_TEST   = 300
TEACHER  = "moonshotai/kimi-k3-free"              # PRD sec.4.1 — dikunci sebagai konstanta
BASE_URL = "https://api.tokenrouter.com/v1"
WORKERS  = 6     # tier gratis: konkurensi tidak dijamin. Turunkan kalau log dipenuhi "HTTP 429".
MAX_SEQ  = 1024                                   # harus sama dengan max_seq_length di notebook 02
STUDENT_TOKENIZER = "Qwen/Qwen3-0.6B"             # untuk statistik panjang token

ROOT = Path("/kaggle/working")
RAW, GEN, PROC, PROMPTS = ROOT/"data/raw", ROOT/"data/generated", ROOT/"data/processed", ROOT/"prompts"
for d in (RAW, GEN, PROC, PROMPTS): d.mkdir(parents=True, exist_ok=True)

try:
    from kaggle_secrets import UserSecretsClient
    API_KEY = UserSecretsClient().get_secret("TOKENROUTER_API_KEY")
except Exception:
    API_KEY = os.environ["TOKENROUTER_API_KEY"]  # fallback kalau run di luar Kaggle

CHAT_URL = f"{BASE_URL}/chat/completions"
HEADERS  = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
USAGE    = {"prompt": 0, "completion": 0}        # tier free bisa lapse -> pantau token terpakai
lock     = threading.Lock()
print("config ok — teacher:", TEACHER, "@", BASE_URL)

config ok — teacher: moonshotai/kimi-k3-free @ https://api.tokenrouter.com/v1


## Subsample GSM8K

1.500 soal train dan 300 soal test, seed tetap. Test diambil dari split test resmi.

In [109]:
from datasets import load_dataset

ds = load_dataset("openai/gsm8k", "main")

def gold(ans: str) -> str:
    """GSM8K menyimpan jawaban final setelah '####'."""
    return ans.split("####")[-1].strip().replace(",", "")

train = ds["train"].shuffle(seed=SEED).select(range(N_TRAIN))
test  = ds["test"].shuffle(seed=SEED).select(range(N_TEST))

train_rows = [{"idx": i, "question": q, "answer": gold(a)}
              for i, (q, a) in enumerate(zip(train["question"], train["answer"]))]
test_rows  = [{"idx": i, "question": q, "answer": gold(a)}
              for i, (q, a) in enumerate(zip(test["question"], test["answer"]))]

def dump(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")

dump(train_rows, RAW/"train_1500.jsonl")
dump(test_rows,  RAW/"test_300.jsonl")
print(len(train_rows), "train /", len(test_rows), "test")
print(train_rows[0]["question"][:200], "\n-> answer:", train_rows[0]["answer"])

1500 train / 300 test
Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found.  How many seashells did Leigh 
-> answer: 16


## Prompt granularity

Satu panggilan menghasilkan G1+G2+G3 sekaligus (PRD sec.5 Tahap 1) — bukan tiga
panggilan terpisah, karena teacher tidak konsisten menjaga jarak antar level
kalau diminta satu-satu, dan biayanya 3x lipat.

Format output pakai **delimiter penanda**, bukan JSON: rationale G3 panjang dan
multi-baris, dan JSON dari LLM rutin rusak karena newline tak ter-escape.
Delimiter tidak punya masalah itu.

In [110]:
PROMPT = r"""You generate training data for a study on chain-of-thought granularity.

For the math problem below, write THREE solutions at three clearly different
levels of detail, then the final answer.

G1 - terse: only the essential arithmetic. No explanatory prose. 1-2 lines.
G2 - moderate: every important calculation, each with a short clause saying what
     it is for. What a good textbook solution looks like.
G3 - verbose: spell out every sub-step. Restate what is given, name what each
     quantity means, show the arithmetic, then verify the result at the end.

Hard requirements:
- All three MUST reach the same final numeric answer.
- Length must clearly increase G1 < G2 < G3, by a wide margin, not a small one.
- The ANSWER section must be a bare number: no units, no commas, no $ sign.
- Use the exact section markers shown. Output nothing outside them.

---- EXAMPLE ----
Problem: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

###G1###
48 / 2 = 24
48 + 24 = 72

###G2###
Natalia sold 48 clips in April. In May she sold half as many, so 48 / 2 = 24 clips.
Adding both months: 48 + 24 = 72 clips altogether.

###G3###
Let us work through what the problem tells us. We are given two months of sales.
In April, Natalia sold clips to 48 of her friends, so April sales = 48 clips.
For May, the problem says she sold half as many clips as in April. "Half as many"
means we divide the April number by 2. So May sales = 48 / 2 = 24 clips.
The question asks for the total across April and May combined, so we add the two
monthly figures together: total = April + May = 48 + 24 = 72 clips.
Let us verify. If May is 24 and that is half of April, then April should be
24 * 2 = 48, which matches the given number. And 48 + 24 = 72, so the total is
consistent. The answer is 72 clips.

###ANSWER###
72
---- END EXAMPLE ----

Problem: {{PROBLEM}}
"""

(PROMPTS/"granularity_prompt.txt").write_text(PROMPT, encoding="utf-8")
print(PROMPT[:400])

You generate training data for a study on chain-of-thought granularity.

For the math problem below, write THREE solutions at three clearly different
levels of detail, then the final answer.

G1 - terse: only the essential arithmetic. No explanatory prose. 1-2 lines.
G2 - moderate: every important calculation, each with a short clause saying what
     it is for. What a good textbook solution looks


## Teacher call + parser

Retry dengan backoff untuk 429/5xx — satu run 1.500 call hampir pasti kena rate
limit setidaknya sekali, dan kehilangan run di tengah jalan lebih mahal daripada
10 baris retry.

In [111]:
SECT = re.compile(r"###\s*(G1|G2|G3|ANSWER)\s*###", re.I)

def call_teacher(problem, retries=5):
    body = {"model": TEACHER,
            "messages": [{"role": "user", "content": PROMPT.replace("{{PROBLEM}}", problem)}],
            "temperature": 0.3, "max_tokens": 1600}
    for i in range(retries):
        try:
            r = requests.post(CHAT_URL, headers=HEADERS, json=body, timeout=300)
            if r.status_code == 200:
                j = r.json()
                u = j.get("usage") or {}
                with lock:
                    USAGE["prompt"]     += u.get("prompt_tokens", 0)
                    USAGE["completion"] += u.get("completion_tokens", 0)
                return j["choices"][0]["message"]["content"]
            if r.status_code not in (408, 429, 500, 502, 503, 520, 524):
                print(f"  HTTP {r.status_code}: {r.text[:160]}", flush=True)
                return None                       # error permanen — jangan retry
            print(f"  HTTP {r.status_code}, retry {i+1}/{retries}", flush=True)
        except requests.RequestException as e:
            print(f"  {type(e).__name__}, retry {i+1}/{retries}", flush=True)
        time.sleep(min(2 ** i, 30) + random.random())   # tier free -> backoff sampai 30s
    return None

def parse_sections(txt):
    """-> {'G1':..,'G2':..,'G3':..,'ANSWER':..} atau None."""
    if not txt: return None
    parts = SECT.split(txt)                        # ['', 'G1', body, 'G2', body, ...]
    if len(parts) < 9: return None
    d = {parts[i].upper(): parts[i + 1].strip() for i in range(1, len(parts) - 1, 2)}
    if not all(k in d and d[k] for k in ("G1", "G2", "G3", "ANSWER")): return None
    d["ANSWER"] = d["ANSWER"].splitlines()[0]   # buang basa-basi setelah angka
    return d

def norm_num(s):
    """Ambil angka terakhir, normalisasi ke float/int. None kalau tak ada angka."""
    nums = re.findall(r"-?\d[\d,]*\.?\d*", str(s).replace("$", ""))
    if not nums: return None
    try: v = float(nums[-1].rstrip(".").replace(",", ""))
    except ValueError: return None
    return int(v) if v == int(v) else round(v, 4)

_t = "###G1###\na\n###G2###\nb\n###G3###\nc\n###ANSWER###\n{}"
assert parse_sections(_t.format("72"))["ANSWER"] == "72"
assert parse_sections(_t.format("72\n\nHope this helps with the other 3!"))["ANSWER"] == "72"
assert parse_sections("###G1###\na\n###G2###\nb\n###ANSWER###\n7") is None   # G3 hilang
assert parse_sections("no markers here") is None
assert norm_num("The answer is $1,200.00") == 1200 and norm_num("abc") is None
print("parser ok")

parser ok


## Diagnostik koneksi

Jalankan ini **sebelum** pilot. Tanpa retry dan timeout pendek, jadi kalau ada
yang salah ketahuan dalam hitungan detik, bukan puluhan menit.

In [112]:
t0 = time.time()
try:
    r = requests.get(f"{BASE_URL}/models", headers=HEADERS, timeout=30)
    print(f"[{time.time()-t0:5.1f}s] GET /models -> {r.status_code}")
    if r.status_code == 200:
        ids = [m.get("id") for m in r.json().get("data", [])]
        print(f"  {len(ids)} model | kimi: {[i for i in ids if 'kimi' in str(i).lower()]}")
    else:
        print("  body:", r.text[:300])
except Exception as e:
    print(f"[{time.time()-t0:5.1f}s] GAGAL {type(e).__name__}: {e}")
    print("  >>> cek Settings -> Internet: ON")

[  0.6s] GET /models -> 200
  121 model | kimi: ['moonshotai/kimi-k3-free', 'moonshotai/kimi-k3', 'moonshotai/kimi-k2.7-code', 'moonshotai/kimi-k2.6', 'moonshotai/kimi-k2.5']


In [113]:
# generasi minimal — memisahkan "koneksi/auth rusak" dari "generasinya lambat"
t0 = time.time()
r = requests.post(CHAT_URL, headers=HEADERS, timeout=60,
                  json={"model": TEACHER, "max_tokens": 5,
                        "messages": [{"role": "user", "content": "Say OK"}]})
print(f"[{time.time()-t0:5.1f}s] HTTP {r.status_code}\n{r.text[:500]}")

[  1.1s] HTTP 200
{"id": "chatcmpl-gw-1b010865ad86afa9b9e652c7", "object": "chat.completion", "created": 1786195037, "model": "kimi-k3", "choices": [{"index": 0, "message": {"role": "assistant", "content": "", "refusal": null, "annotations": null, "audio": null, "function_call": null, "reasoning_content": "The user just asked me"}, "logprobs": null, "finish_reason": "length", "stop_reason": null, "token_ids": null, "routed_experts": null}], "service_tier": null, "system_fingerprint": "vllm-0.1.dev19262+gb6bbf29dd


In [114]:
# ukur latensi prompt sungguhan (1600 token) -> dasar estimasi durasi 1500 call
t0 = time.time()
txt = call_teacher(train_rows[0]["question"])
dt = time.time() - t0
print(f"1 call penuh: {dt:.1f}s | parse: {'ok' if parse_sections(txt) else 'GAGAL'}")
print(f"estimasi {N_TRAIN} call @ {WORKERS} worker: {N_TRAIN*dt/WORKERS/60:.0f} menit")
if dt > 120: print(">>> lambat. Turunkan max_tokens, atau naikkan WORKERS kalau tidak kena 429.")

1 call penuh: 38.1s | parse: ok
estimasi 1500 call @ 6 worker: 159 menit


## Pilot 10 sampel

PRD sec.9: tes prompt pada 10 sampel sebelum membakar 1.500 call. Baca
output-nya — kalau G2 dan G3 terlihat mirip panjangnya, perbaiki prompt di sini,
bukan setelah 1.500 call.

In [115]:
# ok = []
# for r in train_rows[:10]:
#     t0 = time.time()
#     d = parse_sections(call_teacher(r["question"]))
#     print(f"[{r['idx']:2d}] {time.time()-t0:6.1f}s  {'ok' if d else 'GAGAL'}", flush=True)
#     if d: ok.append((r, d))
# print("\nparsed:", len(ok), "/ 10")
# assert ok, "semua call gagal — jalankan cell diagnostik di atas"

# row, d = ok[0]
# for k in ("G1", "G2", "G3"):
#     print(f"\n===== {k} ({len(d[k].split())} kata) =====\n{d[k]}")
# print("\nANSWER:", d["ANSWER"], "| gold:", row["answer"])

In [116]:
# # jarak kasar antar level di pilot — kalau tidak naik tajam, betulkan prompt dulu
# import statistics as st
# for k in ("G1", "G2", "G3"):
#     w = [len(d[k].split()) for _, d in ok]
#     print(f"{k}: mean {st.mean(w):6.1f} kata   min {min(w):4d}  max {max(w):4d}")

## Generasi penuh

Hasil di-append ke JSONL per record. Menjalankan ulang cell ini akan melanjutkan
dari yang sudah berhasil, bukan mengulang dari nol.

In [117]:
# Lanjutkan run sebelumnya: Add Data -> Notebook Output -> notebook ini.
# /kaggle/working selalu mulai kosong, jadi salin dulu hasil lama ke sana.
import shutil, glob
prev = glob.glob("/kaggle/input/*/data/generated/teacher_raw.jsonl")
if prev and not (GEN/"teacher_raw.jsonl").exists():
    shutil.copy(prev[0], GEN/"teacher_raw.jsonl")
    print("disalin dari", prev[0])
else:
    print("mulai dari nol" if not prev else "sudah ada di working")

# Resume mencocokkan lewat idx. Kalau versi `datasets` berubah, shuffle(seed) bisa
# beda -> idx lama menunjuk soal lain, dan rationale menempel ke soal yang salah.
# Tidak akan muncul sebagai error di mana pun, jadi diperiksa di sini.
if (GEN/"teacher_raw.jsonl").exists():
    qmap = {r["idx"]: r["question"] for r in train_rows}
    old = [json.loads(l) for l in open(GEN/"teacher_raw.jsonl", encoding="utf-8") if l.strip()]
    mismatch = [r["idx"] for r in old if qmap.get(r["idx"]) != r["question"]]
    assert not mismatch, (
        f"{len(mismatch)} idx menunjuk soal berbeda dari run sebelumnya. "
        f"Subsample tidak reproducible — HAPUS teacher_raw.jsonl dan generate ulang dari nol.")
    print(f"konsistensi idx: OK ({len(old)} record cocok dengan subsample sekarang)")

mulai dari nol
konsistensi idx: OK (2030 record cocok dengan subsample sekarang)


In [118]:
OUT = GEN/"teacher_raw.jsonl"
done = set()
if OUT.exists():
    with open(OUT, encoding="utf-8") as f:
        for l in f:
            if not l.strip(): continue
            r = json.loads(l)
            if r.get("raw"): done.add(r["idx"])   # raw None = call gagal -> coba lagi
todo = [r for r in train_rows if r["idx"] not in done]
print(f"berhasil {len(done)}, akan dicoba {len(todo)}")

counter = [0]

def work(row):
    txt = call_teacher(row["question"])
    rec = {"idx": row["idx"], "question": row["question"], "gold": row["answer"], "raw": txt}
    with lock:
        with open(OUT, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        counter[0] += 1
        if counter[0] % 25 == 0:
            el = time.time() - t0
            print(f"  {counter[0]}/{len(todo)} | {el/60:.0f}m lewat | "
                  f"sisa ~{el/counter[0]*(len(todo)-counter[0])/60:.0f}m", flush=True)

t0 = time.time()
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    list(ex.map(work, todo))
print(f"selesai dalam {(time.time()-t0)/60:.1f} menit")
print(f"token terpakai — prompt {USAGE['prompt']:,} | completion {USAGE['completion']:,}")
print("cek dashboard TokenRouter: kalau promo free sudah lapse, ini yang dibebankan ke saldo")

berhasil 1499, akan dicoba 1
selesai dalam 3.7 menit
token terpakai — prompt 1,345 | completion 2,079
cek dashboard TokenRouter: kalau promo free sudah lapse, ini yang dibebankan ke saldo


---
# Tahap 2 — Filtering & Validasi

Gate wajib sebelum training. Kalau G2 dan G3 ternyata sama panjang, prompt gagal
dan 14 run berikutnya akan mengukur variabel yang tidak ada.

In [119]:
recs = [json.loads(l) for l in open(OUT, encoding="utf-8") if l.strip()]

# satu idx bisa muncul >1x kalau cell generasi diulang -> ambil yang berhasil
best = {}
for r in recs:
    if r["idx"] not in best or (r.get("raw") and not best[r["idx"]].get("raw")):
        best[r["idx"]] = r
recs = [best[k] for k in sorted(best)]

drop_null = sum(1 for r in recs if not r["raw"])
recs = [r for r in recs if r["raw"]]
parsed, drop_parse, drop_wrong = [], 0, 0

for r in recs:
    d = parse_sections(r["raw"])
    if d is None:
        drop_parse += 1; continue
    if norm_num(d["ANSWER"]) != norm_num(r["gold"]):     # filter kebenaran teacher
        drop_wrong += 1; continue
    parsed.append({"idx": r["idx"], "question": r["question"], "answer": r["gold"],
                   "g1": d["G1"], "g2": d["G2"], "g3": d["G3"]})

with open(GEN/"teacher_parsed.jsonl", "w", encoding="utf-8") as f:
    for p in parsed: f.write(json.dumps(p, ensure_ascii=False) + "\n")

print(f"idx unik      : {len(best)}")
print(f"call gagal    : {drop_null}   <- jalankan ulang cell generasi untuk memulihkan")
print(f"gagal parse   : {drop_parse}")
print(f"jawaban salah : {drop_wrong}")
print(f"lolos         : {len(parsed)}  ({len(parsed)/len(best)*100:.1f}% dari {len(best)})")

idx unik      : 1500
call gagal    : 1   <- jalankan ulang cell generasi untuk memulihkan
gagal parse   : 10
jawaban salah : 34
lolos         : 1455  (97.0% dari 1500)


### Gate: pemisahan panjang token

Diukur dengan tokenizer student (bukan word count) — itu satuan yang benar-benar
dilihat model saat training.

In [120]:
from transformers import AutoTokenizer
import numpy as np, pandas as pd

tok = AutoTokenizer.from_pretrained(STUDENT_TOKENIZER)
L = {k: np.array([len(tok(p[k]).input_ids) for p in parsed]) for k in ("g1", "g2", "g3")}

stats = pd.DataFrame({
    "level": ["G1", "G2", "G3"],
    "mean":  [L[k].mean() for k in L],
    "std":   [L[k].std()  for k in L],
    "p10":   [np.percentile(L[k], 10) for k in L],
    "median":[np.median(L[k]) for k in L],
    "p90":   [np.percentile(L[k], 90) for k in L],
}).round(1)
stats.to_csv(ROOT/"data/processed/token_stats.csv", index=False)
display(stats)

m1, m2, m3 = L["g1"].mean(), L["g2"].mean(), L["g3"].mean()
overlap_12 = np.percentile(L["g1"], 90) >= np.percentile(L["g2"], 10)
overlap_23 = np.percentile(L["g2"], 90) >= np.percentile(L["g3"], 10)
print(f"\nrasio  G2/G1 = {m2/m1:.2f}x   G3/G2 = {m3/m2:.2f}x")
print(f"overlap p90/p10  G1-G2: {overlap_12}   G2-G3: {overlap_23}")

assert m1 < m2 < m3, "GATE GAGAL: urutan panjang tidak monoton — perbaiki prompt"
assert m2/m1 > 1.5 and m3/m2 > 1.5, "GATE GAGAL: level terlalu berdekatan — perbaiki prompt"
print("\nGATE 1 PASS — level terpisah jelas")

,level,mean,std,p10,median,p90
0,G1,33.0,13.5,19.0,30.0,52.0
1,G2,84.9,26.0,56.0,81.0,119.0
2,G3,338.3,68.8,258.0,330.0,432.6


rasio  G2/G1 = 2.57x   G3/G2 = 3.99x
overlap p90/p10  G1-G2: False   G2-G3: False

GATE 1 PASS — level terpisah jelas


### Gate 2: G3 harus muat di `max_seq_len`

Teacher kelas frontier menulis lebih panjang daripada 70B. Kalau `soal + G3 +
jawaban` melebihi 1024 token, TRL memotongnya diam-diam saat training — dan yang
terukur bukan lagi G3, melainkan "G3 terpotong". Itu confound fatal untuk RQ1,
dan tidak akan muncul sebagai error di mana pun.

Cek di sini, saat masih murah untuk diperbaiki.

In [121]:
full_len = np.array([
    len(tok(p["question"]).input_ids) + len(tok(p["g3"]).input_ids)
    + len(tok(str(p["answer"])).input_ids) + 32          # ~32 token overhead chat template
    for p in parsed])

over = (full_len > MAX_SEQ).mean()
print(f"panjang penuh (soal+G3+jawaban): mean {full_len.mean():.0f}  "
      f"p95 {np.percentile(full_len, 95):.0f}  max {full_len.max()}")
print(f"melebihi MAX_SEQ={MAX_SEQ}: {over*100:.1f}% sampel")

if over > 0.02:
    print(f"\n>>> NAIKKAN max_seq_length ke {int(np.percentile(full_len, 99) // 128 + 1) * 128} "
          f"di notebook 02, dan pakai nilai yang sama untuk SEMUA varian "
          f"(kalau beda, itu jadi variabel liar).")
assert over <= 0.10, (
    f"GATE GAGAL: {over*100:.1f}% sampel G3 kena truncate. Naikkan MAX_SEQ "
    f"(biaya: VRAM + waktu) atau perpendek instruksi G3 di prompt.")
print("\nGATE 2 PASS")

panjang penuh (soal+G3+jawaban): mean 432  p95 576  max 805
melebihi MAX_SEQ=1024: 0.0% sampel

GATE 2 PASS


### Bangun G1 / G2 / G3 / AO

Soal identik di keempat varian — ini kontrol eksperimen paling penting di PRD.
Yang berubah hanya isi `rationale`.

In [122]:
for name, key in [("G1", "g1"), ("G2", "g2"), ("G3", "g3"), ("AO", None)]:
    path = PROC/f"train_{name}.jsonl"
    with open(path, "w", encoding="utf-8") as f:
        for p in parsed:
            f.write(json.dumps({"idx": p["idx"], "question": p["question"],
                                "rationale": p[key] if key else "",
                                "answer": p["answer"]}, ensure_ascii=False) + "\n")
    print(name, "->", path.name, len(parsed), "baris")

dump(test_rows, PROC/"test_300.jsonl")

# kontrol: keempat varian harus punya himpunan soal yang persis sama
sets = [{json.loads(l)["question"] for l in open(PROC/f"train_{n}.jsonl", encoding="utf-8")}
        for n in ("G1", "G2", "G3", "AO")]
assert all(s == sets[0] for s in sets), "varian tidak berbagi soal yang sama"
print("\nkontrol soal identik: OK")

G1 -> train_G1.jsonl 1455 baris
G2 -> train_G2.jsonl 1455 baris
G3 -> train_G3.jsonl 1455 baris
AO -> train_AO.jsonl 1455 baris

kontrol soal identik: OK


## Ringkasan dataset

Statistik yang dikutip di seksi Setup README.

In [123]:
summary = pd.DataFrame({
    "Varian": ["G1", "G2", "G3", "AO"],
    "N": [len(parsed)] * 4,
    "Mean tokens (rationale)": [round(m1, 1), round(m2, 1), round(m3, 1), 0.0],
    "Relatif ke G1": ["1.00x", f"{m2/m1:.2f}x", f"{m3/m1:.2f}x", "-"],
})
summary.to_csv(ROOT/"data/processed/dataset_summary.csv", index=False)
display(summary)
print(f"\nteacher: {TEACHER} | yield {len(parsed)}/{N_TRAIN} ({len(parsed)/N_TRAIN*100:.1f}%)")

,Varian,N,Mean tokens (rationale),Relatif ke G1
0,G1,1455,33.0,1.00x
1,G2,1455,84.9,2.57x
2,G3,1455,338.3,10.24x
3,AO,1455,0.0,-


teacher: moonshotai/kimi-k3-free | yield 1455/1500 (97.0%)


---

Output: `data/processed/` berisi `train_G1/G2/G3/AO.jsonl` dan `test_300.jsonl`,
siap dipakai `02_training.ipynb`.